# Step 11 — PI3Kα interaction geometry and provisional lead selection

**Tryptanthrin anticancer capstone | PDB 4JPS | PI3Kα**

This notebook **does not re-dock**. It reuses the saved, previously validated receptor and ten-ligand results, inspects top-three docked poses, and adds auditable geometric screening of hydrogen bonds, hydrophobic proximity and aromatic contacts. It distinguishes *geometrically plausible* interactions from experimentally validated interactions. Results are computational only.

**Required upload:** `PI3Kalpha_docking.zip` (the original handover archive, which contains `06_Molecular_Docking/PI3Kalpha/`). The ZIP renamed folder `PI3K_alpha` is also supported. The notebooks in the ZIP are not rerun.

In [4]:
import io, zipfile, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display

OUT=Path('PI3Kalpha_Step11_Interaction_Review')
OUT.mkdir(exist_ok=True)
uploaded=files.upload()
assert len(uploaded)==1, 'Please upload exactly one PI3Kalpha docking handover ZIP.'
name=next(iter(uploaded))
assert name.lower().endswith('.zip')
z=zipfile.ZipFile(io.BytesIO(uploaded[name]))
assert z.testzip() is None, 'ZIP corruption detected'
archive_files={Path(p).name:p for p in z.namelist() if not p.endswith('/')}
print('ZIP validated:',name,' | ',len(z.namelist()),'entries')
print('Contained folders:',sorted(set('/'.join(p.split('/')[:2]) for p in z.namelist()))[:10])

def load_csv(basename):
    assert basename in archive_files, f'Missing {basename}'
    return pd.read_csv(io.BytesIO(z.read(archive_files[basename])))

def load_text(basename):
    assert basename in archive_files, f'Missing {basename}'
    return z.read(archive_files[basename]).decode('utf-8', errors='replace')

rank=load_csv('PI3Kalpha_final_candidate_ranking.csv')
assert len(rank)==10 and rank['Analog_ID'].nunique()==10
rank=rank.sort_values('Docking_rank_verified').reset_index(drop=True)
top3=rank.head(3).copy()
assert (top3['docking_ok'].astype(str).str.lower()=='true').all()
print('Selected top-three IDs:',top3['Analog_ID'].tolist())
display(top3[['Analog_ID','vina_best_affinity_kcal_mol','Consensus_pIC50','Pred_RF_pIC50','Pred_XGB_pIC50','QED','SA_score','Max_train_Tanimoto']])

Saving PI3Kalpha_docking.zip to PI3Kalpha_docking.zip
ZIP validated: PI3Kalpha_docking.zip  |  83 entries
Contained folders: ['06_Molecular_Docking/PI3Kalpha']
Selected top-three IDs: ['TRY_6db5e7a7802259ab', 'TRY_c88335488290afb4', 'TRY_c6b27d68ce6ec718']


,Analog_ID,vina_best_affinity_kcal_mol,Consensus_pIC50,Pred_RF_pIC50,Pred_XGB_pIC50,QED,SA_score,Max_train_Tanimoto
0,TRY_6db5e7a7802259ab,-10.290,6.29428,6.36380,6.22475,0.540506,2.488290,0.409836
1,TRY_c88335488290afb4,-10.261,6.25409,6.28987,6.21832,0.556121,2.444439,0.403226
2,TRY_c6b27d68ce6ec718,-10.175,6.36582,6.30865,6.42299,0.560274,2.506605,0.413333


## Geometry definitions

* A **plausible hydrogen bond** needs a donor D, its associated explicit hydrogen H, and an acceptor A; thresholds here are D···A ≤3.5 Å, H···A ≤2.7 Å and ∠D–H···A ≥120°. These are screening thresholds, not interaction energies.
* The PDBQT atom types `OA`/`NA` are used to flag oxygen/nitrogen acceptors and `HD` to identify explicit donor hydrogens. Assignment of H to a donor uses a ≤1.3 Å D–H proximity. This is approximate; individual protonation states and tautomerism may differ.
* **Hydrophobic proximity** is carbon-to-carbon separation ≤4.5 Å. It is not, by itself, evidence of a favourable hydrophobic interaction.
* **Aromatic proximity** is a ligand PDBQT `A`-type atom within 5.0 Å of an aromatic receptor-residue carbon; this is *not* proof of π–π stacking because ring-centroid and plane orientations have not been validated.
* All calculations use existing docked coordinates **without alignment or remodelling**.

In [5]:
def parse_pdbqt(text, first_model=False):
    atoms=[]
    for line in text.splitlines():
        if first_model and line.startswith('ENDMDL'):
            break
        if not line.startswith(('ATOM  ','HETATM')):
            continue
        try:
            atoms.append(dict(serial=int(line[6:11]), atom=line[12:16].strip(),
               resname=line[17:20].strip(), chain=line[21:22].strip(),
               resid=line[22:26].strip(),
               x=float(line[30:38]),y=float(line[38:46]),z=float(line[46:54]),
               type=line.split()[-1], xyz=np.array([float(line[30:38]),float(line[38:46]),float(line[46:54])])))
        except (ValueError,IndexError) as e:
            raise ValueError(f'Could not parse PDBQT atom line: {line}') from e
    if not atoms: raise ValueError('PDBQT contains no atom records')
    return atoms

receptor=parse_pdbqt(load_text('4JPS_AB_distance_filtered.pdbqt'))
# IMPORTANT: only Model 1 is analysed for every compound
ligands={}
for aid in top3['Analog_ID']:
    ligands[aid]=parse_pdbqt(load_text(aid+'_docked.pdbqt'),first_model=True)
print('Receptor atoms:',len(receptor))
print('Model 1 ligand atoms:',{k:len(v) for k,v in ligands.items()})
assert all(len(a)>20 for a in ligands.values())

# Pair each explicit polar hydrogen with its nearest O/N donor within 1.3 Å.
def get_dh_pairs(atoms):
    from scipy.spatial import cKDTree
    donors=[a for a in atoms if a['type'] in {'N','OA','NA','NS','OS','SA'} or a['atom'].startswith(('N','O','S'))]
    hs=[a for a in atoms if a['type']=='HD']
    if not donors or not hs:return []
    tree=cKDTree(np.stack([d['xyz'] for d in donors]))
    distances,indices=tree.query(np.stack([h['xyz'] for h in hs]),k=1)
    return [(donors[int(i)],h,float(dist)) for h,dist,i in zip(hs,distances,indices) if dist<=1.3]

def angle_dha(d,h,a):
    v=d['xyz']-h['xyz'];w=a['xyz']-h['xyz']
    denom=float(np.linalg.norm(v)*np.linalg.norm(w))
    if denom<1e-8:return float('nan')
    return float(np.degrees(np.arccos(np.clip(np.dot(v,w)/denom,-1,1))))

def describe(a):
    return f"{a['resname']}:{a['chain']}:{a['resid']}:{a['atom']}"

print('Explicit donor H pairs in receptor:',len(get_dh_pairs(receptor)))
for aid,ats in ligands.items():print(aid,'explicit donor H pairs:',len(get_dh_pairs(ats)))

Receptor atoms: 12336
Model 1 ligand atoms: {'TRY_6db5e7a7802259ab': 28, 'TRY_c88335488290afb4': 30, 'TRY_c6b27d68ce6ec718': 30}
Explicit donor H pairs in receptor: 2262
TRY_6db5e7a7802259ab explicit donor H pairs: 0
TRY_c88335488290afb4 explicit donor H pairs: 2
TRY_c6b27d68ce6ec718 explicit donor H pairs: 1


In [6]:
HB_DA=3.5; HB_HA=2.7; HB_ANG=120.0
hbond_rows=[];hydro_rows=[];arom_rows=[]
protein_acceptors=[x for x in receptor if x['type'] in ('OA','NA')]
protein_dh=get_dh_pairs(receptor)
aromatic_residues={'PHE','TYR','TRP','HIS'}
backbone={'C','CA','N','O','OXT','CB'}

for aid,lig in ligands.items():
    lig_acceptors=[x for x in lig if x['type'] in ('OA','NA')]
    lig_dh=get_dh_pairs(lig)
    for origin,pairs,acceptors in [('ligand_to_protein',lig_dh,protein_acceptors),('protein_to_ligand',protein_dh,lig_acceptors)]:
        for d,h,_ in pairs:
            for a in acceptors:
                da=float(np.linalg.norm(d['xyz']-a['xyz']))
                if da>HB_DA:continue
                ha=float(np.linalg.norm(h['xyz']-a['xyz']))
                ang=angle_dha(d,h,a)
                hbond_rows.append(dict(Analog_ID=aid,orientation=origin,
                    donor=describe(d),hydrogen=describe(h),acceptor=describe(a),
                    donor_acceptor_A=round(da,3),hydrogen_acceptor_A=round(ha,3),
                    D_H_A_angle_deg=round(ang,2),
                    geometry_plausible=bool(ha<=HB_HA and ang>=HB_ANG)))
    lig_carbon=[x for x in lig if x['type'] in ('C','A')]
    prot_carbon=[x for x in receptor if x['type'] in ('C','A')]
    for x in lig_carbon:
        for y in prot_carbon:
            dist=float(np.linalg.norm(x['xyz']-y['xyz']))
            if dist<=4.5:
                hydro_rows.append(dict(Analog_ID=aid,ligand_atom=x['atom'],receptor=describe(y),distance_A=round(dist,3),kind='C-C proximity only'))
            if x['type']=='A' and y['resname'] in aromatic_residues and y['atom'] not in backbone and dist<=5.0:
                arom_rows.append(dict(Analog_ID=aid,ligand_aromatic_atom=x['atom'],receptor=describe(y),distance_A=round(dist,3),kind='aromatic atom proximity; orientation untested'))

hb=pd.DataFrame(hbond_rows,columns=['Analog_ID','orientation','donor','hydrogen','acceptor','donor_acceptor_A','hydrogen_acceptor_A','D_H_A_angle_deg','geometry_plausible'])
hy=pd.DataFrame(hydro_rows,columns=['Analog_ID','ligand_atom','receptor','distance_A','kind'])
ar=pd.DataFrame(arom_rows,columns=['Analog_ID','ligand_aromatic_atom','receptor','distance_A','kind'])
for name,frame in [('PI3Kalpha_hbond_geometry.csv',hb),('PI3Kalpha_hydrophobic_proximity.csv',hy),('PI3Kalpha_aromatic_proximity.csv',ar)]:
    frame.to_csv(OUT/name,index=False)
print('Hydrogen bond candidates (distance-screened):',len(hb),' | passing angle+distance:',int(hb['geometry_plausible'].sum()) if len(hb) else 0)
print('Carbon-pair proximities:',len(hy),' | Aromatic atom proximities:',len(ar))
display(hb.sort_values('donor_acceptor_A').head(20) if len(hb) else hb)

Hydrogen bond candidates (distance-screened): 6  | passing angle+distance: 2
Carbon-pair proximities: 221  | Aromatic atom proximities: 86


,Analog_ID,orientation,donor,hydrogen,acceptor,donor_acceptor_A,hydrogen_acceptor_A,D_H_A_angle_deg,geometry_plausible
2,TRY_c88335488290afb4,protein_to_ligand,TYR:A:836:OH,TYR:A:836:HH,UNL::1:O,2.985,3.628,43.22,False
3,TRY_c6b27d68ce6ec718,ligand_to_protein,UNL::1:N,UNL::1:H,SER:A:854:O,3.074,2.212,141.80,True
4,TRY_c6b27d68ce6ec718,protein_to_ligand,VAL:A:851:N,VAL:A:851:H,UNL::1:O,3.120,2.295,136.10,True
1,TRY_c88335488290afb4,ligand_to_protein,UNL::1:N,UNL::1:H,ASP:A:933:OD1,3.217,3.263,78.47,False
0,TRY_c88335488290afb4,ligand_to_protein,UNL::1:N,UNL::1:H,ASP:A:933:OD1,3.217,3.146,84.87,False
5,TRY_c6b27d68ce6ec718,protein_to_ligand,SER:A:854:OG,SER:A:854:HG,UNL::1:O,3.354,3.107,95.71,False


## Provisional lead selection

The shortlist is **not selected on docking energy alone**. The CSV preserves docking, target-specific ML predictions, RF/XGB agreement, QED, SA, PAINS/Brenk and training-set similarity. The three leading docked compounds are retained as **provisional leads for further investigation**, subject to later medicinal-chemistry review; evidence is insufficient for claims of measured affinity or activity. Geometric interactions are annotations, not extra points in an invented combined score.

In [7]:
leads=top3.copy()
for aid in leads['Analog_ID']:
    x=hb[hb.Analog_ID==aid]
    idx=leads.index[leads['Analog_ID']==aid][0]
    leads.loc[idx,'HB_geometry_plausible_count']=int(x['geometry_plausible'].sum()) if len(x) else 0
    leads.loc[idx,'hydrophobic_carbon_pair_count']=int((hy.Analog_ID==aid).sum())
    leads.loc[idx,'aromatic_atom_proximity_count']=int((ar.Analog_ID==aid).sum())
    leads.loc[idx,'selection_status']='Provisional: integrated ML + Vina + QED/SA/alerts; low applicability-domain similarity'
    leads.loc[idx,'interaction_caution']='Plausible D-H-A geometry only; hydrophobic/aromatic proximity not confirmed energetic interaction'
lead_columns=['Docking_rank_verified','Analog_ID','SMILES','vina_best_affinity_kcal_mol',
 'Consensus_pIC50','Pred_RF_pIC50','Pred_XGB_pIC50','Model_disagreement_abs',
 'Max_train_Tanimoto','QED','SA_score','PAINS_flag','Brenk_flag',
 'HB_geometry_plausible_count','hydrophobic_carbon_pair_count','aromatic_atom_proximity_count',
 'selection_status','interaction_caution']
leads[lead_columns].to_csv(OUT/'PI3Kalpha_final_provisional_lead_selection.csv',index=False)
display(leads[lead_columns])
assert len(leads)==3 and leads['Analog_ID'].nunique()==3

lines=['# PI3Kα — Step 11 Interaction Geometry and Provisional Lead Review','',
 'Reference: 4JPS; the receptor and analogue docked poses are unchanged from the original handover.',
 'This analysis uses explicit polar hydrogens retained in PDBQT and no coordinate alignment.',
 '','## Criteria',
 'Hydrogen-bond geometry: D···A <=3.5 Å, H···A <=2.7 Å, D–H···A >=120°. Proximity is not experimental confirmation.',
 'Hydrophobic contacts: ligand/protein carbon-pair distance <=4.5 Å; preliminary proximity only.',
 'Aromatic proximity: ligand aromatic-typed atom within 5.0 Å of aromatic-residue side-chain carbon; π-stacking orientation not established.',
 '','## Provisional shortlist']
for _,row in leads.iterrows():
    lines.append(f"- {row['Analog_ID']}: Vina {row['vina_best_affinity_kcal_mol']:.3f} kcal/mol; consensus predicted pIC50 {row['Consensus_pIC50']:.3f}; QED {row['QED']:.3f}; SA {row['SA_score']:.2f}; maximum training Tanimoto {row['Max_train_Tanimoto']:.3f}; geometrically plausible D–H···A contacts {int(row['HB_geometry_plausible_count'])}.")
lines+=['','## Limitations',
 '- Predictions are computational and do not establish PI3Kα inhibition or anticancer activity.',
 '- Vina score differences are small and not equivalent to experimental affinity differences.',
 '- Protein/ligand protonation, hydrogen assignment and tautomeric states may alter hydrogen-bond geometry.',
 '- Carbon proximity does not establish favourable hydrophobic energy; aromatic geometry needs centroid/plane-level review.',
 '- Low similarity to ML training structures limits confidence in predicted pIC50.',
 '- Further structural checks, chemical synthesis feasibility and experimental testing are needed.',
 '- Existing original workflow validation RMSD was 0.6039 Å; no receptor or docking parameters were altered in this step.']
(OUT/'PI3Kalpha_step11_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Saved report and 4 CSV files to',OUT)

,Docking_rank_verified,Analog_ID,SMILES,vina_best_affinity_kcal_mol,Consensus_pIC50,Pred_RF_pIC50,Pred_XGB_pIC50,Model_disagreement_abs,Max_train_Tanimoto,QED,SA_score,PAINS_flag,Brenk_flag,HB_geometry_plausible_count,hydrophobic_carbon_pair_count,aromatic_atom_proximity_count,selection_status,interaction_caution
0,1,TRY_6db5e7a7802259ab,O=C1c2cc(C3CC3)ccc2-n2c1nc1cc(N3CCOCC3)ccc1c2=O,-10.290,6.29428,6.36380,6.22475,0.13905,0.409836,0.540506,2.488290,False,False,0.0,77.0,23.0,Provisional: integrated ML + Vina + QED/SA/ale...,Plausible D-H-A geometry only; hydrophobic/aro...
1,2,TRY_c88335488290afb4,NC(=O)c1ccc2c(c1)C(=O)c1nc3cc(N4CCOCC4)ccc3c(=...,-10.261,6.25409,6.28987,6.21832,0.07155,0.403226,0.556121,2.444439,False,False,0.0,67.0,24.0,Provisional: integrated ML + Vina + QED/SA/ale...,Plausible D-H-A geometry only; hydrophobic/aro...
2,3,TRY_c6b27d68ce6ec718,CC(=O)Nc1ccc2c(c1)-n1c(nc3c(N4CCOCC4)cccc3c1=O...,-10.175,6.36582,6.30865,6.42299,0.11434,0.413333,0.560274,2.506605,False,False,2.0,77.0,39.0,Provisional: integrated ML + Vina + QED/SA/ale...,Plausible D-H-A geometry only; hydrophobic/aro...


Saved report and 4 CSV files to PI3Kalpha_Step11_Interaction_Review


In [8]:
# Download a small output ZIP — add these files to your existing repo after inspecting them.
import zipfile
zip_path=Path('PI3Kalpha_Step11_Interaction_Review.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as output_zip:
    for file in sorted(OUT.iterdir()):output_zip.write(file,arcname=file.name)
with zipfile.ZipFile(zip_path) as test:assert test.testzip() is None
print('ZIP contents:',[p.name for p in OUT.iterdir()]);files.download(str(zip_path))

ZIP contents: ['PI3Kalpha_hbond_geometry.csv', 'PI3Kalpha_step11_report.md', 'PI3Kalpha_hydrophobic_proximity.csv', 'PI3Kalpha_aromatic_proximity.csv', 'PI3Kalpha_final_provisional_lead_selection.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>